**ENCODER**

Lets build a FeedForwardSubLayer for our encoder-only transformer. This layer will consist of two linear layers with a ReLU activation function between them. It also takes two parameters, d_model and d_ff, which represent the dimensionality of the input embeddings and the dimension between the linear layers, respectively.

In [9]:
import torch.nn as nn
import torch
import math
import torch.nn.functional as F 

In [6]:
class FeedForwardSubLayer(nn.Module):
    def __init__(self, d_model, d_ff):
        super().__init__()
        # Define the layers and activation
        self.fc1 = nn.Linear(d_model, d_ff)
        self.fc2 = nn.Linear(d_ff, d_model)
        self.relu = nn.ReLU()

    def forward(self, x):
        # Pass the input through the layers and activation
        return self.fc2(self.relu(self.fc1(x)))
    
# Instantiate the FeedForwardSubLayer and apply it to x
feed_forward = FeedForwardSubLayer(512,2048)

# Define a sample input tensor 'x'
x = torch.randn(32,10, 512) # Example: batch size 1, d_model 512

output = feed_forward(x)
print(f"Input shape: {x.shape}")
print(f"Output shape: {output.shape}")

Input shape: torch.Size([32, 10, 512])
Output shape: torch.Size([32, 10, 512])


 As we can see, the input and output shapes are the same, which is expected because the first linear layer projects the embeddings into a higher-dimensional space, the activation function introduces non-linearity, and the second linear layer projects the features back into the original dimensions. Let's integrate this into an encoder layer

With a FeedForwardSubLayer class defined, we have all of the pieces you need to define an EncoderLayer class. Note that the encoder layer typically consists of a multi-head attention mechanism, and a feed-forward sublayer with layer normalization and dropout on the sublayer's inputs and outputs.

In [15]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        assert d_model % num_heads == 0, "d_model must be divisible by num_heads."        
        self.num_heads = num_heads        
        self.d_model = d_model        
        self.head_dim = d_model // num_heads        
        self.query_linear = nn.Linear(d_model, d_model, bias=False)        
        self.key_linear = nn.Linear(d_model, d_model, bias=False)
        self.value_linear = nn.Linear(d_model, d_model, bias=False)
        self.output_linear = nn.Linear(d_model, d_model)
        
    def split_heads(self, x, batch_size):
        seq_length = x.size(1)
        x = x.reshape(batch_size, seq_length, self.num_heads, self.head_dim)
        return x.permute(0, 2, 1, 3)
    
    def compute_attention(self, query, key, value, mask=None):
        scores = torch.matmul(query, key.transpose(-2, -1)) / (self.head_dim ** 0.5)
        if mask is not None:
            scores = scores.masked_fill(mask == 0, float('-inf'))
        attention_weights = F.softmax(scores, dim=-1)
        return torch.matmul(attention_weights, value)

    def combine_heads(self, x, batch_size):        
        x = x.permute(0, 2, 1, 3).contiguous()
        return x.view(batch_size, -1, self.d_model)
    
    def forward(self, query, key, value, mask=None):
        batch_size = query.size(0)
        query = self.split_heads(self.query_linear(query), batch_size)
        key = self.split_heads(self.key_linear(key), batch_size)
        value = self.split_heads(self.value_linear(value), batch_size)
        attention_weights = self.compute_attention(query, key, value, mask)
        output = self.combine_heads(attention_weights, batch_size)
        return self.output_linear(output)

In [16]:
class EncoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout):
        super().__init__()
        # Instantiate the layers
        self.self_attn = MultiHeadAttention(d_model, num_heads)
        self.ff_sublayer = FeedForwardSubLayer(d_model, d_ff)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, src_mask):
        # Complete the forward method
        attn_output = self.self_attn(x,x,x,src_mask)
        x = self.norm1(x + self.dropout(attn_output))
        ff_output = self.ff_sublayer(x)
        x = self.norm2(x + self.dropout(ff_output))
        return x

 From here, we're only one final step away from completing the encoder transformer body

In [17]:
class InputEmbeddings(nn.Module):
    def __init__(self, vocab_size: int, d_model: int) -> None:
        super().__init__()
        # Set the model dimensionality and vocabulary size
        self.d_model = d_model
        self.vocab_size = vocab_size
        # Instantiate the embedding layer
        self.embedding = nn.Embedding(vocab_size,d_model)

    def forward(self, x):
        # Return the embeddings multiplied by the square root of d_model
        return self.embedding(x) * math.sqrt(self.d_model)
    
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_seq_length):
        super().__init__()
        # Create a matrix of zeros of dimensions max_seq_length by d_model
        pe = torch.zeros(max_seq_length, d_model)
        position = torch.arange(0, max_seq_length, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * -(math.log(10000.0) / d_model))
        
        # Perform the sine and cosine calculations
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        # Ensure pe isn't a learnable parameter during training
        self.register_buffer('pe', pe.unsqueeze(0))
        
    def forward(self, x):
        # Add the positional embeddings to the token embeddings
        return x + self.pe[:, :x.size(1)]


In [18]:
class TransformerEncoder(nn.Module):
    def __init__(self, vocab_size, d_model, num_layers, num_heads, d_ff, dropout, max_seq_length):
        super().__init__()
        # Define the embedding, positional encoding, and encoder layers
        self.embedding = InputEmbeddings(vocab_size, d_model)
        self.positional_encoding = PositionalEncoding(d_model, max_seq_length)
        self.layers = nn.ModuleList([EncoderLayer(d_model, num_heads, d_ff, dropout) for _ in range(num_layers)])

    def forward(self, x, src_mask):
        # Perform the forward pass through the layers
        x = self.embedding(x)
        x = self.positional_encoding(x)
        for layer in self.layers:
            x = layer(x, src_mask)
        return x

Once trained, the encoder body will encode the input sequences into rich embeddings that capture complex patterns in the sequence. To convert these embeddings into a prediction, you need to add a head to your body. Lets build a classification head and test out passing it some token IDs

Lets design a transformer head that could be used for classification tasks like sentiment analysis or categorization. We'll define a ClassifierHead class, create instances of the body and head, and pass a series of token IDs through them both to test that they work as expected.

Note: because this model has been trained yet, the outputs will be meaningless, but testing the code can process inputs and generate outputs in the form we expect is a good test.

In [27]:
vocab_size = 10000
d_model = 512
num_layers = 6
num_heads = 8
d_ff = 2048
dropout = 0.1
seq_length = 256
num_classes = 3
batch_size = 8
max_seq_length = 20

In [22]:
# Complete the classification head
class ClassifierHead(nn.Module):
    def __init__(self, d_model, num_classes):
        super().__init__()
        self.fc = nn.Linear(d_model, num_classes)

    def forward(self, x):
        logits = self.fc(x)
        return F.log_softmax(logits, dim=-1)
      
# Instantiate the encoder transformer's body and head
encoder = TransformerEncoder(vocab_size, d_model, num_layers, num_heads, d_ff, dropout, seq_length)
classifier = ClassifierHead(d_model, num_classes)

# Create dummy input_sequence and src_mask for demonstration
input_sequence = torch.randint(0, vocab_size, (batch_size, seq_length))
src_mask = torch.ones(batch_size, 1, seq_length) # A simple mask, assuming no padding initially

# Complete the forward pass
output = encoder(input_sequence, src_mask)
classification = classifier(output) 
print(f"Classification outputs for a batch of {batch_size} sequences:\n{classification}")
print(f"Encoder output shape: {output.shape}\nClassification head output shape: {classification.shape}")

Classification outputs for a batch of 8 sequences:
tensor([[[-1.9235, -0.5954, -1.1954],
         [-1.8193, -0.6156, -1.2121],
         [-2.0159, -0.2416, -2.5083],
         ...,
         [-0.7527, -1.2149, -1.4603],
         [-1.0131, -0.6889, -2.0041],
         [-0.9396, -1.4266, -0.9966]],

        [[-1.0012, -0.8356, -1.6146],
         [-2.0040, -0.3989, -1.6392],
         [-2.1455, -0.2174, -2.5461],
         ...,
         [-1.0476, -0.6214, -2.1889],
         [-1.6889, -0.9939, -0.8093],
         [-0.9884, -1.2598, -1.0667]],

        [[-1.8518, -0.6204, -1.1865],
         [-2.0214, -0.6617, -1.0454],
         [-1.7985, -1.1451, -0.6612],
         ...,
         [-1.9677, -0.5432, -1.2753],
         [-1.0391, -0.8295, -1.5609],
         [-1.9458, -0.8012, -0.8956]],

        ...,

        [[-1.6743, -0.9228, -0.8791],
         [-1.6195, -0.6011, -1.3713],
         [-1.2792, -0.7524, -1.3843],
         ...,
         [-1.4494, -0.9003, -1.0248],
         [-1.0085, -0.8530, -1.5650],

As we can see from the output shapes, the classification head has successfully reduced the output embeddings into classification predictions. Until trained these predictions are meaningless, but it's a good test that the class works as expected

**DECODER**

Designing a mask for self-attention
To ensure that the decoder can learn to predict tokens, it's important to mask future tokens when modeling the input sequences. We'll build a mask in the form of a triangular matrix of True and False values, with False values in the upper diagonal to exclude future tokens.

In [32]:
seq_length= 10

# Create a Boolean matrix to mask future tokens
tgt_mask = (1 - torch.triu(
  torch.ones(1, seq_length, seq_length), diagonal=1)
).bool()

print(tgt_mask)

tensor([[[ True, False, False, False, False, False, False, False, False, False],
         [ True,  True, False, False, False, False, False, False, False, False],
         [ True,  True,  True, False, False, False, False, False, False, False],
         [ True,  True,  True,  True, False, False, False, False, False, False],
         [ True,  True,  True,  True,  True, False, False, False, False, False],
         [ True,  True,  True,  True,  True,  True, False, False, False, False],
         [ True,  True,  True,  True,  True,  True,  True, False, False, False],
         [ True,  True,  True,  True,  True,  True,  True,  True, False, False],
         [ True,  True,  True,  True,  True,  True,  True,  True,  True, False],
         [ True,  True,  True,  True,  True,  True,  True,  True,  True,  True]]])


Like encoder transformers, decoder transformers are also built of multiple layers that make use of multi-head attention and feed-forward sublayers. 

In [33]:
class DecoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, num_heads)
        self.ff_sublayer = FeedForwardSubLayer(d_model, d_ff)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, tgt_mask):
        # Perform the attention calculation
        attn_output = self.self_attn(x,x,x,tgt_mask)
        # Apply dropout and the first layer normalization
        x = self.norm1(x + self.dropout(attn_output))
        # Pass through the feed-forward sublayer
        ff_output = self.ff_sublayer(x)
        # Apply dropout and the second layer normalization
        x = self.norm2(x + self.dropout(ff_output))
        return x

Time to build the decoder transformer body. This will mean combining the InputEmbeddings, PositionalEncoding, and DecoderLayer classes we've created previously

In [48]:
class TransformerDecoder(nn.Module):
    def __init__(self, vocab_size, d_model, num_layers, num_heads, d_ff, dropout, max_seq_length):
        super(TransformerDecoder, self).__init__()
        self.embedding = InputEmbeddings(vocab_size, d_model)
        self.positional_encoding = PositionalEncoding(d_model, max_seq_length)
        # Define the list of decoder layers and linear layer
        self.layers = nn.ModuleList([DecoderLayer(d_model, num_heads, d_ff, dropout) for _ in range(num_layers)])
        # Define a linear layer to project hidden states to likelihoods
        self.fc = nn.Linear(d_model,vocab_size)

    def forward(self, x, encoder_output, tgt_mask, cross_mask):
        # Complete the forward pass
        x = self.embedding(x)
        x = self.positional_encoding(x)
        for layer in self.layers:
            x = layer(x, encoder_output, tgt_mask, cross_mask)
        x = self.fc(x)
        return F.log_softmax(x, dim=-1)

input_tokens = torch.tensor([[6044, 8239, 4933, 3760, 8963, 8379, 5427, 8503, 3497, 5683], [4101, 6866, 2756, 1399, 5878,  376,   56, 9868, 8794, 6033]])
# Instantiate a decoder transformer and apply it to input_tokens and tgt_mask
transformer_decoder = TransformerDecoder(vocab_size, d_model, num_layers, num_heads, d_ff, dropout, max_seq_length)

The shape [2, 10, 10000] matches [batch_size, seq_length, vocab_size], so the decoder transformer successfully projected the embeddings into token probabilities. Remember, because this model hasn't been trained, these probabilities won't be meaningful. Let's now learn how to combine the decoder and encoder transformers into the encoder-decoder transformer that was shown in the Attention Is All You Need paper.

**ENCODER-DECODER**

To integrate the encoder and decoder stacks you've defined previously into an encoder-decoder transformer, you need to create a cross-attention mechanism to act as a bridge between the two.

In [49]:
class DecoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, num_heads)
        # Define cross-attention and a third layer normalization
        self.cross_attn = MultiHeadAttention(d_model, num_heads)
        self.ff_sublayer = FeedForwardSubLayer(d_model, d_ff)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, y, tgt_mask, cross_mask):
        self_attn_output = self.self_attn(x, x, x, tgt_mask)
        x = self.norm1(x + self.dropout(self_attn_output))
        # Complete the forward pass
        cross_attn_output = self.cross_attn(x,y,y, cross_mask)
        x = self.norm2(x + self.dropout(cross_attn_output))
        ff_output = self.ff_sublayer(x)
        x = self.norm3(x + self.dropout(ff_output))
        return x

Now that we've updated the DecoderLayer class, and the equivalent changes have been made to TransformerDecoder, we're ready to put everything together. Because we've built your classes in a modular and hierarchical way, we only need to call instantiate two of them to build the encoder-decoder transformer: TransformerDecoder and TransformerEncoder

In [50]:
class Transformer(nn.Module):
    def __init__(self, vocab_size, d_model, num_heads, num_layers, d_ff, max_seq_length, dropout):
        super().__init__()
        self.encoder = TransformerEncoder(vocab_size, d_model, num_layers, num_heads, d_ff, dropout, max_seq_length)
        self.decoder = TransformerDecoder(vocab_size, d_model, num_layers, num_heads, d_ff, dropout, max_seq_length)

    def forward(self, x, src_mask, tgt_mask, cross_mask):
        # Complete the forward pass
        encoder_output = self.encoder(x, src_mask)
        decoder_output = self.decoder(x, encoder_output, tgt_mask, cross_mask)
        return decoder_output
cross_mask = torch.tensor([[1, 1, 1, 1, 1, 0, 1, 1, 1, 1],[0, 1, 1, 0, 0, 1, 0, 0, 0, 0],[1, 1, 0, 0, 1, 0, 1, 1, 1, 1],[0, 0, 0, 1, 0, 1, 1, 1, 0, 1],[0, 0, 1, 0, 1, 1, 0, 0, 1, 0],[1, 0, 1, 0, 1, 0, 1, 0, 0, 0],[1, 0, 1, 0, 1, 0, 0, 0, 0, 0],[1, 0, 0, 1, 0, 0, 0, 1, 0, 0],[1, 0, 1, 0, 0, 1, 1, 0, 0, 0],[1, 1, 0, 0, 0, 0, 0, 1, 0, 1]])
input_tokens = torch.tensor([[6044, 8239, 4933, 3760, 8963, 8379, 5427, 8503, 3497, 5683],[4101, 6866, 2756, 1399, 5878,  376,   56, 9868, 8794, 6033]])
src_mask = torch.tensor([[0, 1, 1, 0, 0, 1, 1, 1, 1, 0],
        [1, 0, 1, 0, 1, 1, 0, 1, 1, 0],
        [0, 1, 0, 1, 1, 1, 1, 1, 0, 1],
        [0, 1, 1, 1, 1, 0, 1, 0, 0, 1],
        [1, 0, 1, 0, 1, 0, 0, 0, 0, 0],
        [1, 1, 0, 0, 0, 1, 1, 0, 1, 0],
        [0, 1, 0, 1, 1, 1, 1, 1, 1, 0],
        [1, 1, 0, 0, 1, 0, 0, 1, 1, 0],
        [1, 0, 0, 1, 0, 0, 0, 1, 1, 0],
        [1, 0, 0, 0, 0, 0, 1, 0, 1, 0]])
# Instantiate and call the transformer
transformer = Transformer(vocab_size, d_model, num_heads, num_layers, d_ff, max_seq_length, dropout)

outputs = transformer(input_tokens, src_mask, tgt_mask, cross_mask)
print(outputs)
print(outputs.shape)

tensor([[[ -9.6109, -10.6089,  -9.9542,  ...,  -9.5037,  -8.4172,  -9.7246],
         [ -9.7140,  -9.7140, -10.0871,  ..., -10.5221,  -9.9953,  -9.7188],
         [ -9.0664,  -9.7458, -10.9132,  ..., -10.3987,  -8.6384, -10.1410],
         ...,
         [ -9.6205, -10.1292, -11.6342,  ..., -10.3241,  -9.0082,  -9.4030],
         [ -9.2714,  -9.3283, -10.0156,  ..., -10.7615,  -9.2005,  -9.3026],
         [ -9.2073, -10.2153, -10.7316,  ..., -10.4007,  -9.5391,  -9.1336]],

        [[ -9.5553,  -9.6768,  -9.7287,  ...,  -9.8882,  -9.3311,  -9.5012],
         [ -9.9015,  -9.1040,  -8.9799,  ..., -10.1605,  -8.3823, -10.0429],
         [ -9.1760,  -9.8894,  -9.8186,  ...,  -8.9152,  -9.5092,  -9.5857],
         ...,
         [ -8.7371, -10.1441, -10.1410,  ...,  -9.6720,  -9.9480,  -7.6483],
         [ -8.8490,  -8.7145,  -8.9476,  ...,  -9.7268,  -9.1881,  -8.8340],
         [ -8.7248, -10.4676, -10.2019,  ...,  -8.9783,  -9.5220,  -8.5805]]],
       grad_fn=<LogSoftmaxBackward0>)
torch.

We've built an encoder-decoder transformer from end to end with PyTorch.